In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder

In [2]:
BASE_ENV = Path().resolve().parent

In [3]:
dos_df = pd.read_csv(BASE_ENV / 'data/raw/cicids_wednesday.csv')

In [4]:
dos_df.columns = dos_df.columns.str.strip()
dos_df.columns = dos_df.columns.str.replace(' ', '_')
dos_df.columns = dos_df.columns.str.replace('/', '_')

In [5]:
dos_df = dos_df[dos_df['Label'].str.contains('DoS', case=False)] 

dos_df.loc[dos_df['Label'].str.contains('DoS', case=False, na=False), 'Label'] = 'DoS'

In [6]:
dos_df = dos_df.sample(n=50000, random_state=42)

In [7]:
rename_map = {
    'Total_Fwd_Packets':        'Total_Fwd_Packet',
    'Total_Backward_Packets':   'Total_Bwd_packets',
    'Min_Packet_Length':        'Packet_Length_Min',
    'Max_Packet_Length':        'Packet_Length_Max',
    'Avg_Fwd_Segment_Size':     'Fwd_Segment_Size_Avg',
    'Avg_Bwd_Segment_Size':     'Bwd_Segment_Size_Avg',
    'Init_Win_bytes_forward':   'FWD_Init_Win_Bytes',
    'Init_Win_bytes_backward':  'Bwd_Init_Win_Bytes',
    'act_data_pkt_fwd':         'Fwd_Act_Data_Pkts',
    'min_seg_size_forward':     'Fwd_Seg_Size_Min',
}

In [8]:
dos_df = dos_df.rename(columns=rename_map)

In [9]:
# drop the columns with infinite and nan values
dos_df = dos_df.replace([np.inf, -np.inf], np.nan)
dos_df = dos_df.dropna(axis=1)

In [10]:
dos_df['FWD_Init_Win_Bytes'] = dos_df['FWD_Init_Win_Bytes'].replace(-1, 0)
dos_df['Bwd_Init_Win_Bytes'] = dos_df['Bwd_Init_Win_Bytes'].replace(-1, 0)

negative_cols = dos_df.select_dtypes(include=[np.number]).columns
dos_df = dos_df[(dos_df[negative_cols] >= 0).all(axis=1)]

In [11]:
dos_df = dos_df.drop(columns='Fwd_Header_Length.1')

In [12]:
# load column names
df = pd.read_csv(BASE_ENV / 'data/raw/CICFlowMeter_out.csv', encoding='utf-8')

In [13]:
columns_to_drop = ['Fwd_URG_Flags',
                    'ECE_Flag_Count',
                    'Bwd_URG_Flags',
                    'Fwd_Packet_Bulk_Avg',
                    'Fwd_Bulk_Rate_Avg',
                    'CWR_Flag_Count',
                    'Fwd_Bytes_Bulk_Avg',
                    'URG_Flag_Count',
                    'Flow_ID',
                    'Src_IP',
                    'Src_Port',
                    'Dst_IP',
                    'Dst_Port',
                    'Fwd_Packet_Length_Std',
                    'SYN_Flag_Count',
                    'Packet_Length_Variance',
                    'Flow_IAT_Min',
                    'Subflow_Fwd_Bytes',
                    'Active_Std',
                    'Active_Max',
                    'Idle_Min',
                    'Subflow_Fwd_Packets',
                    'Active_Min',
                    'Fwd_PSH_Flags',
                    'Subflow_Bwd_Packets',
                    'Idle_Std',
                    'Subflow_Bwd_Bytes',
                    'Idle_Max',
                    'Active_Mean',
                    'RST_Flag_Count',
                    'Idle_Mean',
                    'Bwd_PSH_Flags',
                    'Protocol'
                    
 ]
                

In [14]:
df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace(' ', '_')
df.columns = df.columns.str.replace('/', '_')
df.columns

Index(['Flow_ID', 'Src_IP', 'Src_Port', 'Dst_IP', 'Dst_Port', 'Protocol',
       'Timestamp', 'Flow_Duration', 'Total_Fwd_Packet', 'Total_Bwd_packets',
       'Total_Length_of_Fwd_Packet', 'Total_Length_of_Bwd_Packet',
       'Fwd_Packet_Length_Max', 'Fwd_Packet_Length_Min',
       'Fwd_Packet_Length_Mean', 'Fwd_Packet_Length_Std',
       'Bwd_Packet_Length_Max', 'Bwd_Packet_Length_Min',
       'Bwd_Packet_Length_Mean', 'Bwd_Packet_Length_Std', 'Flow_Bytes_s',
       'Flow_Packets_s', 'Flow_IAT_Mean', 'Flow_IAT_Std', 'Flow_IAT_Max',
       'Flow_IAT_Min', 'Fwd_IAT_Total', 'Fwd_IAT_Mean', 'Fwd_IAT_Std',
       'Fwd_IAT_Max', 'Fwd_IAT_Min', 'Bwd_IAT_Total', 'Bwd_IAT_Mean',
       'Bwd_IAT_Std', 'Bwd_IAT_Max', 'Bwd_IAT_Min', 'Fwd_PSH_Flags',
       'Bwd_PSH_Flags', 'Fwd_URG_Flags', 'Bwd_URG_Flags', 'Fwd_Header_Length',
       'Bwd_Header_Length', 'Fwd_Packets_s', 'Bwd_Packets_s',
       'Packet_Length_Min', 'Packet_Length_Max', 'Packet_Length_Mean',
       'Packet_Length_Std', 'Packet_Len

In [15]:

# 2. Split the dataset into two temporary DataFrames
df_normal = df[df['Label'] == 'Benign']
df_attacks = df[df['Label'] != 'Benign']

# Calculate total number of attacks (should be around 22,215 based on your image)
total_attacks = len(df_attacks)

print(f"Original Normal count: {len(df_normal)}")
print(f"Original Attack count: {total_attacks}")

# 3. Undersample the Normal traffic to a 2:1 ratio
# (If you want a 1:1 ratio, just remove the * 2)
df_normal_undersampled = df_normal.sample(n=total_attacks * 2, random_state=42)

# 4. Recombine the datasets into your new, balanced master DataFrame
df_balanced = pd.concat([df_normal_undersampled, df_attacks])

# 5. Shuffle the rows thoroughly so the model doesn't read all normal packets first

Original Normal count: 3450658
Original Attack count: 89583


In [16]:
df = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

In [17]:
shared_cols = [col for col in df.columns if col in dos_df.columns]
dos_df = dos_df[shared_cols]
# Fill missing columns with 0
for col in df.columns:
    if col not in dos_df.columns and col != 'Label':
        dos_df[col] = 0

# Verify
print(f"DoS df shape after filling: {dos_df.shape}")
print(f"Your df shape: {df.shape}")

DoS df shape after filling: (49966, 84)
Your df shape: (268749, 84)


In [18]:
df = df.drop(columns=[c for c in columns_to_drop if c in df.columns])
dos_df = dos_df.drop(columns=[c for c in columns_to_drop if c in dos_df.columns])
df = pd.concat([df, dos_df], ignore_index=True)

In [19]:
df['Label'].value_counts()

Label
Benign            179166
DoS                54433
Exploits           30951
Fuzzers            29613
Reconnaissance     16735
Generic             4632
Shellcode           2102
Backdoor             452
Analysis             385
Worms                246
Name: count, dtype: int64

In [20]:
merge_map = {
    'Analysis': 'Rare_Attack',
    'Backdoor': 'Rare_Attack',
    'Worms' : 'Rare_Attack',
    'Generic' : 'Rare_Attack', 
    'Shellcode' : 'Rare_Attack'
}
df['Label'] = df['Label'].replace(merge_map)

In [21]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'], infer_datetime_format=True, errors='coerce')


/tmp/ipykernel_32635/1390217211.py:1: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df['Timestamp'] = pd.to_datetime(df['Timestamp'], infer_datetime_format=True, errors='coerce')
/tmp/ipykernel_32635/1390217211.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Timestamp'] = pd.to_datetime(df['Timestamp'], infer_datetime_format=True, errors='coerce')


In [22]:
def temporal_split(data, train_ratio=0.70, val_ratio=0.10):
    n = len(data)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    return (
        data.iloc[:train_end],
        data.iloc[train_end:val_end],
        data.iloc[val_end:]
    )

In [23]:
# Do temporal split on original df first
benign_df = df[df['Label'] == 'Benign'].sort_values('Timestamp')
attack_df = df[df['Label'] != 'Benign'].sort_values('Timestamp')

# Remove dos_df from df BEFORE temporal split
df_no_dos = df[df['Label'] != 'DoS']

benign_df = df_no_dos[df_no_dos['Label'] == 'Benign'].sort_values('Timestamp')
attack_df = df_no_dos[df_no_dos['Label'] != 'Benign'].sort_values('Timestamp')

benign_train, benign_val, benign_test = temporal_split(benign_df)
attack_train, attack_val, attack_test = temporal_split(attack_df)

# Then inject DoS manually with proper ratio
dos_train = dos_df.sample(frac=0.70, random_state=42)
dos_val   = dos_df.drop(dos_train.index).sample(frac=0.33, random_state=42)
dos_test  = dos_df.drop(dos_train.index).drop(dos_val.index)

train_df = pd.concat([benign_train, attack_train, dos_train]).sample(frac=1, random_state=42)
val_df   = pd.concat([benign_val,   attack_val,   dos_val]).sample(frac=1, random_state=42)
test_df  = pd.concat([benign_test,  attack_test,  dos_test]).sample(frac=1, random_state=42)

In [24]:
for split in [train_df, val_df, test_df]:
    split.drop(columns='Timestamp', errors='ignore', inplace=True)

X_train, y_train = train_df.drop(columns=['Label']), train_df['Label']
X_val,   y_val   = val_df.drop(columns=['Label']),   val_df['Label']
X_test,  y_test  = test_df.drop(columns=['Label']),  test_df['Label']

In [25]:
print(f"Train size: {len(X_train)} | Val size: {len(X_val)} | Test size: {len(X_test)}")
print(f"Total: {len(X_train) + len(X_val) + len(X_test)}")

print("\n--- Train Attack Distribution ---")
print(y_train.value_counts())

print("\n--- Val Attack Distribution ---")
print(y_val.value_counts())

print("\n--- Test Attack Distribution ---")
print(y_test.value_counts())

Train size: 219973 | Val size: 31374 | Test size: 62901
Total: 314248

--- Train Attack Distribution ---
Label
Benign            125416
DoS                34976
Exploits           21539
Fuzzers            20987
Reconnaissance     11661
Rare_Attack         5394
Name: count, dtype: int64

--- Val Attack Distribution ---
Label
Benign            17916
DoS                4947
Exploits           3174
Fuzzers            2844
Reconnaissance     1710
Rare_Attack         783
Name: count, dtype: int64

--- Test Attack Distribution ---
Label
Benign            35834
DoS               10043
Exploits           6238
Fuzzers            5782
Reconnaissance     3364
Rare_Attack        1640
Name: count, dtype: int64


In [26]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_val_scaled = scaler.transform(X_val)


In [27]:
encoder = LabelEncoder()
y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)
y_val_encoded = encoder.transform(y_val)

In [28]:
ART = BASE_ENV / 'artifacts'

In [29]:
# ART = BASE_ENV / "artifacts"

# train
np.save(ART / "datasets/train/X.npy", X_train_scaled)
np.save(ART / "datasets/train/y.npy", y_train_encoded)

# test
np.save(ART / "datasets/test/X.npy", X_test_scaled)
np.save(ART / "datasets/test/y.npy", y_test_encoded)

# val
np.save(ART / "datasets/val/X.npy", X_val_scaled)
np.save(ART / "datasets/val/y.npy", y_val_encoded)

# scaler
joblib.dump(scaler, ART / "preprocessors/scaler.joblib")

# encoder
joblib.dump(encoder, ART / "preprocessors/encoder.joblib")

['/mnt/c/Users/hassa/OneDrive/Documents/Programming/Python/projects/heimdall/Heimdall/ai-service/IDS/artifacts/preprocessors/encoder.joblib']